<a href="https://colab.research.google.com/github/mikhail-mat/mit-ocw_hands-on-deep-learning/blob/main/Bag_Of_Words_Lyrics_Genre_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

In [2]:
keras.utils.set_random_seed(42)

## Playing around with vectorizing text

The text vectorization layer performs the STIE process
1) Standardise the input text: convert to lowercase and remove punctuation
2) Tokenise the text, split it into tokens by whitespace, so each word is a token
3) Index the tokens, index 0 is UNK, index 1 is the most frequent word in the text, and so on
4) Encode the tokens using multi-hot encoding, so the output is a matrix of 0s and 1s with the number of columns equal to the size of the vocabulary

In [3]:
text_vectorization = keras.layers.TextVectorization(
    output_mode='multi_hot',
    standardize='lower_and_strip_punctuation',
    split='whitespace'
)

In [4]:
dataset = [
    "I tried so hard and got so far, ",
    "But in the end, it doesn't even matter. ",
    "I had to fall to lose it all, "
    "But in the end, it doesn't even matter. "
]

In [5]:
text_vectorization.adapt(dataset)

In [6]:
vocab = text_vectorization.get_vocabulary()

In [7]:
len(vocab)

21

In [8]:
pd.DataFrame(vocab)

,0
0,[UNK]
1,it
2,to
3,the
4,so
5,matter
6,in
7,i
8,even
9,end


In [9]:
test_sentence = "I tried so hard and got to the end"
encoded_sentence = text_vectorization(test_sentence)
print(encoded_sentence)

tf.Tensor([0 0 1 1 1 0 0 1 0 1 0 0 1 0 1 0 1 0 0 1 0], shape=(21,), dtype=int64)


## Getting the data

In [3]:
train_url = "https://www.dropbox.com/scl/fi/ito6bnl2yaf1uw0uqibzf/lyric_genre_train.csv?rlkey=04dkn5un2djza8x0bdmfnlw3u&st=y47qh8i4&dl=1"
val_url = "https://www.dropbox.com/scl/fi/xmywjzqsaa8n5sn1bs0t9/lyric_genre_val.csv?rlkey=hggbeo0s1iaxjpa6z80429xl9&st=6i7d8eau&dl=1"
test_url = "https://www.dropbox.com/scl/fi/fnocl69w9ojs9s5zb0xvf/lyric_genre_test.csv?rlkey=z4hjopw7vaihoh948cbb5mvdp&st=xwond7dp&dl=1"

train_df = pd.read_csv(train_url,index_col=0)
val_df = pd.read_csv(val_url,index_col=0)
test_df = pd.read_csv(test_url,index_col=0)

print(f"""
Train samples: {train_df.shape[0]}
Validation samples: {val_df.shape[0]}
Test samples: {test_df.shape[0]}
      """)


Train samples: 48991
Validation samples: 16331
Test samples: 21774
      


In [11]:
train_df.head()

,Lyric,Genre
0,"Oh, girl. I can't get ready (Can't get ready f...",Pop
1,We met on a rainy evening in the summertime. D...,Pop
2,We carried you in our arms. On Independence Da...,Rock
3,I know he loved you. A long time ago. I ain't ...,Pop
4,Paralysis through analysis. Yellow moral uncle...,Rock


Baseline model would have 55% accuracy if we predict rock every time

In [12]:
train_df['Genre'].value_counts() / train_df.shape[0]

,count
Genre,
Rock,0.549448
Pop,0.295136
Hip Hop,0.155416


Two ways to do one-hot-encoding

In [13]:
# from sklearn.preprocessing import OneHotEncoder
# ohe = OneHotEncoder()
# y_train = ohe.fit_transform(train_df['Genre'])
# y_val = ohe.fit_transform(val_df['Genre'])
# y_test = ohe.fit_transform(test_df['Genre'])

In [4]:
y_train = pd.get_dummies(train_df['Genre']).to_numpy(dtype='float32')
y_val = pd.get_dummies(val_df['Genre']).to_numpy(dtype='float32')
y_test = pd.get_dummies(test_df['Genre']).to_numpy(dtype='float32')

In [15]:
y_train

array([[0., 1., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       ...,
       [0., 1., 0.],
       [0., 1., 0.],
       [0., 0., 1.]], dtype=float32)

## Bag of words model

In [16]:
max_tokens = 5000 # take the 5000 most frequent words
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='multi_hot'
)

In [17]:
text_vectorization.adapt(train_df['Lyric'])

In [18]:
text_vectorization.get_vocabulary()[:20]

['[UNK]',
 np.str_('the'),
 np.str_('you'),
 np.str_('i'),
 np.str_('to'),
 np.str_('and'),
 np.str_('a'),
 np.str_('me'),
 np.str_('it'),
 np.str_('my'),
 np.str_('in'),
 np.str_('im'),
 np.str_('on'),
 np.str_('your'),
 np.str_('that'),
 np.str_('of'),
 np.str_('all'),
 np.str_('be'),
 np.str_('is'),
 np.str_('we')]

In [19]:
text_vectorization.get_vocabulary()[-20:]

[np.str_('eden'),
 np.str_('dagger'),
 np.str_('curve'),
 np.str_('cheddar'),
 np.str_('brew'),
 np.str_('appears'),
 np.str_('vacant'),
 np.str_('universal'),
 np.str_('unholy'),
 np.str_('terrified'),
 np.str_('stickin'),
 np.str_('rumble'),
 np.str_('rug'),
 np.str_('pam'),
 np.str_('os'),
 np.str_('ooohh'),
 np.str_('motto'),
 np.str_('marshall'),
 np.str_('loyalty'),
 np.str_('legacy')]

In [20]:
X_train = text_vectorization(train_df['Lyric'])
X_val = text_vectorization(val_df['Lyric'])
X_test = text_vectorization(test_df['Lyric'])

In [21]:
X_train

<tf.Tensor: shape=(48991, 5000), dtype=int64, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]])>

In [22]:
input = keras.Input(shape=(5000,))
hidden = keras.layers.Dense(4, activation='relu')(input)
output = keras.layers.Dense(3, activation='softmax')(hidden)

model = keras.Model(input, output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 5000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │        20,004 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            15 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,019 (78.20 KB)

 Trainable params: 20,019 (78.20 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [24]:
model.fit(X_train,
          y_train,
          validation_data=(X_val, y_val),
          epochs=10,
          batch_size=32)

Epoch 1/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.6853 - loss: 0.7456 - val_accuracy: 0.7504 - val_loss: 0.5799
Epoch 2/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.7662 - loss: 0.5436 - val_accuracy: 0.7474 - val_loss: 0.5815
Epoch 3/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7790 - loss: 0.5124 - val_accuracy: 0.7467 - val_loss: 0.5909
Epoch 4/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.7884 - loss: 0.4937 - val_accuracy: 0.7444 - val_loss: 0.6019
Epoch 5/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.7957 - loss: 0.4796 - val_accuracy: 0.7427 - val_loss: 0.6129
Epoch 6/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8002 - loss: 0.4688 - val_accuracy: 0.7405 - val_loss: 0.6235
Epoch 7/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8048 - loss: 0.4601 - val_accuracy: 0.7382 - val_loss: 0.6332
Epoch 8/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.8084 - loss: 0.4530

In [25]:
model.evaluate(X_test, y_test)

681/681 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7313 - loss: 0.6666


[0.6715866327285767, 0.7303205728530884]

## Trying bigrams instead of individual words as tokens

In [5]:
bigram_text_vectorization = keras.layers.TextVectorization(
    ngrams=2,
    max_tokens=10000,
    output_mode='multi_hot'
)

In [6]:
bigram_text_vectorization.adapt(train_df['Lyric'])

In [7]:
X_train = bigram_text_vectorization(train_df['Lyric'])
X_val = bigram_text_vectorization(val_df['Lyric'])
X_test = bigram_text_vectorization(test_df['Lyric'])

In [8]:
input = keras.Input(shape=(10000,))
x = keras.layers.Dropout(0.5)(input)
x = keras.layers.Dense(8, activation='relu')(x)
output = keras.layers.Dense(3, activation='softmax')(x)

model = keras.Model(input, output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │        80,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,035 (312.64 KB)

 Trainable params: 80,035 (312.64 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.6628 - loss: 0.7436 - val_accuracy: 0.7479 - val_loss: 0.5878
Epoch 2/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.7476 - loss: 0.5909 - val_accuracy: 0.7523 - val_loss: 0.5767
Epoch 3/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7557 - loss: 0.5633 - val_accuracy: 0.7500 - val_loss: 0.5782
Epoch 4/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7608 - loss: 0.5453 - val_accuracy: 0.7507 - val_loss: 0.5780
Epoch 5/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7674 - loss: 0.5363 - val_accuracy: 0.7515 - val_loss: 0.5733
Epoch 6/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7717 - loss: 0.5242 - val_accuracy: 0.7528 - val_loss: 0.5727
Epoch 7/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7771 - loss: 0.5109 - val_accuracy: 0.7553 - val_loss: 0.5696
Epoch 8/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7800 - loss: 0.5056 -

In [11]:
model.evaluate(X_test, y_test)

681/681 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7528 - loss: 0.5818


[0.5905284285545349, 0.7497014999389648]